# 🔤 Python String DP — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> String DP is like using a mirror on a sentence. You ask: which parts look the same
> read forwards and backwards? Each substring is a small puzzle piece — and you build
> the answer for longer substrings from shorter ones you've already solved.
> The key is always: what is true about s[i..j] given what you know about s[i+1..j-1]?

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is String DP? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Patterns](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Longest Palindromic Substring (LC 5)](#5) |
| 6 | [Pattern 2: Longest Palindromic Subsequence (LC 516)](#6) |
| 7 | [Pattern 3: Palindromic Substrings Count (LC 647)](#7) |
| 8 | [Pattern 4: Longest Common Subsequence Revisited (LC 1143)](#8) |
| 9 | [The String DP Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is String DP? The Visual Model

```
               STRING DP — THE MIRROR TABLE

  INTERVAL DP — fill a 2D table where dp[i][j] represents s[i..j]

  For s = "babad":
  is_palindrome[i][j] — True if s[i..j] is a palindrome

       b  a  b  a  d
  b  [ T  F  T  F  F ]
  a  [ .  T  F  T  F ]
  b  [ .  .  T  F  F ]
  a  [ .  .  .  T  F ]
  d  [ .  .  .  .  T ]

  Fill ORDER: diagonal by diagonal (length 1, then 2, then 3...)
  RULE: s[i][j] is palindrome ↔ s[i]==s[j] AND (length≤2 OR s[i+1][j-1] is palindrome)

  EXPAND-AROUND-CENTER (faster for substring):
  For each center (single char or between chars), expand outward while chars match.
  O(n²) time, O(1) space — no DP table needed.

  TWO TECHNIQUES:
  1. Expand from center: O(n²) time, O(1) space — for longest palindromic SUBSTRING
  2. Interval DP table: O(n²) time, O(n²) space — for palindromic SUBSEQUENCES and counts
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# INTERVAL DP TABLE — for palindrome problems
def build_palindrome_table(s):
    n = len(s)
    # dp[i][j] = True if s[i..j] is a palindrome
    dp = [[False] * n for _ in range(n)]

    # all single chars are palindromes (length 1)
    for i in range(n):
        dp[i][i] = True

    # fill by increasing length
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1        # right boundary
            if s[i] == s[j]:
                if length == 2:
                    dp[i][j] = True   # "aa" — two matching chars
                else:
                    dp[i][j] = dp[i+1][j-1]  # inner substring must also be palindrome
    return dp

s = "babad"
table = build_palindrome_table(s)
print("palindrome table for 'babad':")
for i, row in enumerate(table):
    print(f"  {s[i]}: {row}")

# EXPAND FROM CENTER — for longest palindromic substring
def expand_around_center(s, left, right):
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left  -= 1
        right += 1
    return s[left+1:right]   # the palindrome found (undo the last failed expansion)

print("\nexpand from center 'bab' in 'babad':", expand_around_center("babad", 1, 1))
print("expand from center 'aba' in 'babad':", expand_around_center("babad", 1, 2))
print("String DP setup demonstrated.")

<a id='3'></a>
## 3. The Core API — All Patterns

```
STRING DP PATTERN          TECHNIQUE              COMPLEXITY
────────────────────────────────────────────────────────────────────
Longest palindromic substr  Expand around center   O(n²) time, O(1) space
Count palindromic substrs   Expand around center   O(n²) time, O(1) space
Longest palindromic subseq  Interval DP (2D)        O(n²) time, O(n²) space
Min cuts for palindrome     DP + palindrome table   O(n²) time
Longest common subsequence  2D DP (match/skip)      O(m*n) time, O(n) space
Edit distance               2D DP (del/ins/rep)     O(m*n) time, O(n) space

SUBSTRING vs SUBSEQUENCE:
  Substring:    contiguous characters — use expand-from-center or sliding window
  Subsequence:  any characters in order (can skip) — use 2D DP on two strings

THINGS YOU DO NOT DO:
❌  Use interval DP for substring problems (expand-from-center is O(1) space)
❌  Forget to handle even-length palindromes (center is between two chars)
❌  Mix up dp[i][j] row-fill order in interval DP — always fill shorter lengths first
❌  Access dp[i+1][j-1] when j-i < 2 (undefined for length-1 or length-2 strings)
```

In [ ]:
# DEMO: two centers for expand-around-center
# odd-length palindrome: center = single character
# even-length palindrome: center = gap between two characters

def all_palindromes(s):
    result = []
    n = len(s)
    for center in range(n):
        # odd-length: single char center
        p = expand_around_center(s, center, center)
        if len(p) > 1:
            result.append(p)
        # even-length: gap between center and center+1
        if center + 1 < n:
            p = expand_around_center(s, center, center + 1)
            if len(p) > 1:
                result.append(p)
    return result

print("all multi-char palindromes in 'aacaa':", all_palindromes("aacaa"))
print("all multi-char palindromes in 'abba':",  all_palindromes("abba"))
print("Expand-around-center demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"longest palindromic substring"         Expand from center: O(n²) O(1)
"count palindromic substrings"          Expand from center, count each
"longest palindromic subsequence"       Interval DP: dp[i][j]=dp[i+1][j-1]+2 if match
"minimum cuts for palindrome partition" DP + precompute palindrome table
"longest common subsequence"            2D DP match/skip
"edit distance"                         2D DP del/ins/replace
"distinct subsequences"                 2D DP counting
"interleaving strings"                  2D DP bool
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Longest Palindromic Substring — LC 5

---

```
PROBLEM:
  Given a string s, return the longest palindromic substring.

TRICK:
  Expand around every possible center. For n characters there are 2n-1 centers
  (n single-char + n-1 gaps between chars).
  For each center, expand outward as long as s[left] == s[right].
  Track the longest palindrome found.

SLOW MOTION TRACE on s="cbbd":

  center=0 (c): expand c→c only → "c" (len 1)
  center=0-1 gap (c/b): c≠b → "" (len 0)
  center=1 (b): expand b→b only → "b" (len 1)
  center=1-2 gap (b/b): b==b → "bb" (len 2) ← new best!
    expand left=0,right=3: c≠d → stop
  center=2 (b): b → "b" (len 1)
  center=2-3 gap (b/d): b≠d → ""
  center=3 (d): d → "d" (len 1)

  longest = "bb" at positions [1,2]

KEY INSIGHT:
  2n-1 centers covers ALL palindromes (odd-length and even-length).
  Each expansion takes O(n) worst case → total O(n²).
  No DP table needed — O(1) space.

TIME:  O(n²)
SPACE: O(1)
```

In [ ]:
def longest_palindrome(s):
    """
    LC 5 — Longest Palindromic Substring
    Approach: Expand around each of the 2n-1 centers; track best start/end indices.
    Args:
        s (str): input string.
    Returns:
        str: the longest palindromic substring.
    Time:  O(n²) — n centers, each expands O(n) in worst case
    Space: O(1)  — only track indices, no DP table
    """
    if not s:
        return ""

    best_start, best_len = 0, 1   # track by index/length (avoids string slicing in loop)

    def expand(left, right):
        nonlocal best_start, best_len
        while left >= 0 and right < len(s) and s[left] == s[right]:
            left  -= 1
            right += 1
        # after loop: left+1..right-1 is the palindrome
        plen = right - left - 1
        if plen > best_len:
            best_len   = plen
            best_start = left + 1   # +1 because we overshot

    for i in range(len(s)):
        expand(i, i)      # odd-length: single character center
        expand(i, i + 1)  # even-length: gap between i and i+1

    return s[best_start : best_start + best_len]

# Slow motion on "cbbd":
# center 0: expand(0,0) → 'c' len=1; expand(0,1) → c≠b → len=0
# center 1: expand(1,1) → 'b' len=1; expand(1,2) → b==b → expand l=0,r=3 → c≠d → len=2 best!
# center 2: expand(2,2) → 'b' len=1; expand(2,3) → b≠d → len=0
# center 3: expand(3,3) → 'd' len=1
# return s[1:3] = 'bb'

def test_harness(fn):
    def is_valid_longest_palindrome(s, result):
        if result not in s: return False
        if result != result[::-1]: return False
        # check no longer palindrome exists
        for length in range(len(result)+1, len(s)+1):
            for start in range(len(s)-length+1):
                sub = s[start:start+length]
                if sub == sub[::-1]: return False
        return True

    tests = ["babad", "cbbd", "a", "ac", "racecar", "abacaba", "aaaa"]
    passed = 0
    for s in tests:
        got = fn(s)
        ok = is_valid_longest_palindrome(s, got)
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | s='{s}' | got='{got}'")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")

test_harness(longest_palindrome)
print("longest_palindrome defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Longest Palindromic Subsequence — LC 516

---

```
PROBLEM:
  Given a string s, find the length of the longest palindromic subsequence.
  A subsequence can skip characters (not necessarily contiguous).

TRICK:
  Interval DP: dp[i][j] = length of longest palindromic subsequence in s[i..j].
  If s[i] == s[j]: dp[i][j] = dp[i+1][j-1] + 2   (both endpoints contribute)
  Else:            dp[i][j] = max(dp[i+1][j], dp[i][j-1])  (skip one endpoint)
  Base: dp[i][i] = 1 (single character is a palindrome of length 1).
  Fill by increasing substring length.

  EQUIVALENT TRICK: LPS(s) = LCS(s, reverse(s))

SLOW MOTION TRACE on s="bbbab":

  dp (5×5), fill length=1,2,3,4,5:

  len=1: dp[i][i]=1 for all i
  len=2: b==b→2, b==b→2, b≠a→1, a≠b→1
  len=3: dp[0][2]: b==b→dp[1][1]+2=3; dp[1][3]: b≠a→max(dp[2][3],dp[1][2])=max(1,2)=2
  len=4: dp[0][3]: b≠a→max(dp[1][3],dp[0][2])=max(2,3)=3; dp[1][4]: b==b→dp[2][3]+2=3
  len=5: dp[0][4]: b==b→dp[1][3]+2=2+2=4

  answer = dp[0][4] = 4 ("bbbb" is the LPS)

KEY INSIGHT:
  Interval DP fills the table diagonally. Always requires shorter subproblems first.
  s[i]==s[j] → use both + inner; else discard one endpoint and take the best.

TIME:  O(n²)
SPACE: O(n²) full table, reducible to O(n) with rolling arrays
```

In [ ]:
def longest_palindrome_subseq(s):
    """
    LC 516 — Longest Palindromic Subsequence
    Approach: Interval DP — dp[i][j] = LPS length in s[i..j]; fill by increasing length.
    Args:
        s (str): input string.
    Returns:
        int: length of the longest palindromic subsequence.
    Time:  O(n²) — fill each cell of the n×n table once
    Space: O(n²) — dp table (reducible to O(n) with 2-row rolling)
    """
    n = len(s)
    dp = [[0] * n for _ in range(n)]

    for i in range(n):
        dp[i][i] = 1          # every single character is a palindrome of length 1

    for length in range(2, n + 1):       # fill by increasing length
        for i in range(n - length + 1):
            j = i + length - 1
            if s[i] == s[j]:
                inner = dp[i+1][j-1] if length > 2 else 0  # guard: inner doesn't exist for len=2
                dp[i][j] = inner + 2      # both endpoints + whatever's inside
            else:
                dp[i][j] = max(dp[i+1][j], dp[i][j-1])  # skip left or skip right endpoint

    return dp[0][n-1]

# Slow motion on "bbbab" (n=5):
# len=1: dp[0][0]=dp[1][1]=dp[2][2]=dp[3][3]=dp[4][4]=1
# len=2:
#   (0,1): b==b → dp[0][1]=0+2=2
#   (1,2): b==b → dp[1][2]=2
#   (2,3): b≠a → max(dp[3][3],dp[2][2])=1 → dp[2][3]=1
#   (3,4): a≠b → max(dp[4][4],dp[3][3])=1 → dp[3][4]=1
# len=5: (0,4): b==b → dp[1][3]+2=2+2=4
# return dp[0][4]=4

def test_harness(fn):
    tests = [
        ("bbbab", 4),
        ("cbbd", 2),
        ("a", 1),
        ("ac", 1),
        ("racecar", 7),  # whole string is palindrome
        ("abcba", 5),
        ("agbdba", 5),   # "abdba"
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | s='{inputs[0]}' | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(longest_palindrome_subseq)
print("longest_palindrome_subseq defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Palindromic Substrings Count — LC 647

---

```
PROBLEM:
  Given a string s, return the number of palindromic substrings.
  Every single character counts as a palindromic substring.

TRICK:
  Same expand-around-center technique as LC 5.
  For each center, every successful expansion = one more palindromic substring.
  Count += 1 for each (left, right) pair where expansion succeeds.

SLOW MOTION TRACE on s="aaa":

  center 0 (a): expand(0,0) → a: count+1=1; expand(0,1) → a==a: count+1=2; l=-1 stop
  center 1 (a): expand(1,1) → a: count+1=3; expand(1,2) → a==a: count+1=4; l=0,r=3 → stop
    also expand(0,2): a==a → count+1=5; l=-1 → stop (this expansion came from center 1)
  center 2 (a): expand(2,2) → a: count+1=6; expand(2,3) → out of bounds

  Wait: recounting — let me be precise:
  center 0 odd: a         → count 1
  center 0-1 even: aa     → count 1
  center 1 odd: a, aaa    → count 2 (single a, then expand to aaa)
  center 1-2 even: aa     → count 1
  center 2 odd: a         → count 1
  Total = 1+1+2+1+1 = 6

KEY INSIGHT:
  Each center expansion that succeeds = one palindrome found.
  Count in a while loop: count += 1 each iteration.

TIME:  O(n²)
SPACE: O(1)
```

In [ ]:
def count_substrings(s):
    """
    LC 647 — Palindromic Substrings
    Approach: Expand from each of 2n-1 centers; count every successful expansion.
    Args:
        s (str): input string.
    Returns:
        int: total count of palindromic substrings.
    Time:  O(n²) — 2n-1 centers, each expands up to O(n)
    Space: O(1)  — just a counter
    """
    count = 0
    n = len(s)

    def expand_count(left, right):
        nonlocal count
        while left >= 0 and right < n and s[left] == s[right]:
            count  += 1     # this (left, right) span is a palindrome → count it
            left   -= 1
            right  += 1

    for i in range(n):
        expand_count(i, i)      # odd-length: center at i
        expand_count(i, i + 1)  # even-length: center between i and i+1

    return count

# Slow motion on "aaa":
# expand_count(0,0): a→count=1; OOB
# expand_count(0,1): a==a→count=2; l=-1 stop
# expand_count(1,1): a→count=3; expand l=0,r=2: a==a→count=4; l=-1 stop
# expand_count(1,2): a==a→count=5; l=0,r=3: OOB stop
# expand_count(2,2): a→count=6
# expand_count(2,3): OOB immediately
# return 6

def test_harness(fn):
    tests = [
        ("abc", 3),    # a, b, c
        ("aaa", 6),    # a,a,a,aa,aa,aaa
        ("aba", 4),    # a,b,a,aba
        ("a", 1),
        ("racecar", 10),
        ("fdsklf", 6), # only single chars
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | s='{inputs[0]}' | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_substrings)
print("count_substrings defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Minimum Insertions for Palindrome — LC 1143 Extension

---

```
PROBLEM (extension of LCS):
  Minimum insertions to make a string palindrome.
  This equals: n - LPS(s) where LPS = Longest Palindromic Subsequence.

  Also reviewed here: LPS via LCS trick — LPS(s) = LCS(s, reverse(s))

TRICK:
  The characters you DON'T need to insert are those already forming the LPS.
  Characters outside the LPS need to be mirrored (= inserted on the other side).
  min_insertions = n - LPS(s) = n - LCS(s, reversed(s))

SLOW MOTION TRACE on s="mbadm":

  reversed(s) = "mdabm"
  LCS("mbadm", "mdabm"):

         m  d  a  b  m
  m  [0][1][1][1][1][1]
  b  [0][1][1][1][2][2]
  a  [0][1][1][2][2][2]
  d  [0][1][2][2][2][2]
  m  [0][1][2][2][2][3]

  LCS = 3 ("mam" or "mdm" etc.)
  min_insertions = 5 - 3 = 2 (insert 'd' and 'b' to make "mbdadm" or similar)

KEY INSIGHT:
  min insertions = n - LPS is a beautiful reduction.
  LPS = LCS(s, s[::-1]) is an easy implementation trick.

TIME:  O(n²)
SPACE: O(n) rolling
```

In [ ]:
def min_insertions_palindrome(s):
    """
    Min insertions to make s a palindrome.
    Approach: LPS via LCS(s, reversed(s)); answer = n - LPS.
    Args:
        s (str): input string.
    Returns:
        int: minimum characters to insert to make s a palindrome.
    Time:  O(n²) — LCS on two strings of length n
    Space: O(n)  — rolling 1D array
    """
    t = s[::-1]      # reverse — LCS(s, reverse) = LPS(s)
    n = len(s)
    prev = [0] * (n + 1)

    for i in range(1, n + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if s[i-1] == t[j-1]:
                curr[j] = prev[j-1] + 1          # characters match — extend LCS
            else:
                curr[j] = max(prev[j], curr[j-1]) # skip one from either side
        prev = curr

    lps = prev[n]                    # LCS(s, reverse) = LPS(s)
    return n - lps                   # characters outside LPS need to be inserted

# Also show the direct interval DP approach for comparison
def min_insertions_interval(s):
    """
    Min insertions via interval DP: min_ins[i][j] = min inserts to make s[i..j] palindrome.
    Time: O(n²)  Space: O(n²)
    """
    n = len(s)
    dp = [[0]*n for _ in range(n)]
    for length in range(2, n+1):
        for i in range(n-length+1):
            j = i + length - 1
            if s[i] == s[j]:
                dp[i][j] = dp[i+1][j-1] if length > 2 else 0  # endpoints match freely
            else:
                dp[i][j] = min(dp[i+1][j], dp[i][j-1]) + 1    # insert to mirror one end
    return dp[0][n-1]

# Slow motion on 'mbadm':
# reverse = 'mdabm'
# LCS('mbadm','mdabm')=3 (e.g. 'mdm' or 'mam')
# min_insertions = 5-3 = 2

def test_harness(fn):
    tests = [
        ("mbadm", 2),
        ("leetcode", 5),
        ("zjveiiwvc", 4),
        ("a", 0),
        ("aa", 0),
        ("ab", 1),
        ("abcba", 0),  # already palindrome
    ]
    passed = 0
    for *inputs, expected in tests:
        got_lcs = min_insertions_palindrome(inputs[0])
        got_ivl = min_insertions_interval(inputs[0])
        ok = (got_lcs == expected) and (got_ivl == expected)
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | s='{inputs[0]}' | expected={expected} | LCS={got_lcs} interval={got_ivl}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")

test_harness(min_insertions_palindrome)
print("min_insertions_palindrome and min_insertions_interval defined.")

<a id='9'></a>
## 9. The String DP Decision Map

```
QUESTION TYPE                          TECHNIQUE                      LC
─────────────────────────────────────────────────────────────────────────
Longest palindromic substring          Expand from center             5
Count palindromic substrings           Expand, count per expansion    647
Longest palindromic subsequence        Interval DP or LCS(s,rev)      516
Min insertions for palindrome          n - LPS                        1312
Longest common subsequence             2D DP match/skip               1143
Edit distance                          2D DP del/ins/replace          72
Distinct subsequences                  2D DP counting                 115
Interleaving strings                   2D DP bool                     97

SUBSTRING vs SUBSEQUENCE (critical distinction):
  Substring  = contiguous → expand-from-center, O(1) space
  Subsequence = skip allowed → 2D DP, O(n²) space

INTERVAL DP FILL ORDER:
  for length in range(2, n+1):      # OUTER: short to long
    for i in range(n-length+1):     # INNER: left boundary
      j = i + length - 1            # right boundary
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for String DP:**

| Signal | What to Do |
|--------|------------|
| "longest palindromic substring" | Expand from center, track best |
| "count palindromic substrings" | Expand from center, count every hit |
| "longest palindromic subsequence" | Interval DP or LCS(s, reversed) |
| "min insertions for palindrome" | n − LPS |
| "two strings, common structure" | LCS or edit distance 2D DP |

**2. The core templates — memorize these:**

```python
# EXPAND FROM CENTER
for i in range(n):
    expand(i, i)      # odd length
    expand(i, i+1)    # even length

def expand(l, r):
    while l >= 0 and r < n and s[l] == s[r]:
        # record palindrome s[l:r+1]
        l -= 1; r += 1

# INTERVAL DP (fill by length)
dp = [[0]*n for _ in range(n)]
for i in range(n): dp[i][i] = 1
for length in range(2, n+1):
    for i in range(n-length+1):
        j = i + length - 1
        if s[i] == s[j]:
            dp[i][j] = (dp[i+1][j-1] if length>2 else 0) + 2
        else:
            dp[i][j] = max(dp[i+1][j], dp[i][j-1])

# LPS = LCS(s, reversed(s))
lps = lcs(s, s[::-1])
min_insertions = n - lps
```

**4. Gotchas to not forget:**

```
❌  Only expanding for odd-length centers — always add even-length (gap) centers
❌  Accessing dp[i+1][j-1] for length-2 substrings — guard with if length>2
❌  Filling interval DP by row — must fill by INCREASING LENGTH (diagonal)
❌  Confusing substring (contiguous) with subsequence (can skip) — different algorithms
✅  Expand-from-center: O(1) space, works for both substring and counting
✅  LPS(s) = LCS(s, s[::-1]) — easy to implement if LCS is already coded
✅  min_insertions = n - LPS — one-liner once you have LPS
✅  Every single character is a palindromic substring (count them too!)
```

## Summary Map

```
                    🔤 STRING DP
                         │
           ┌─────────────┼─────────────┐
           │             │             │
       SUBSTRING     SUBSEQUENCE    TWO-STRING
       (contiguous)  (can skip)     (alignment)
           │             │             │
     EXPAND FROM    INTERVAL DP     LCS / Edit
     CENTER         len 1→2→...n    Distance
     O(n²) O(1)sp   O(n²) O(n²)sp  O(mn) O(n)sp
           │             │
    ┌──────┴──┐     dp[i][j]=
    │         │     s[i]==s[j]?
 LONGEST   COUNT    diag+2 : max
 LC 5      LC 647   LC 516
                         │
                   MIN INSERTS
                   n - LPS

CORE RULE:
  Substring → expand from center.
  Subsequence → interval DP or LCS(s, reversed).
  Fill interval DP by length (short→long), never by row.
```

---
*End of String DP Master Guide — Sean Edition*